# Deep Learning Assignment 1 - Manual Backpropagation

## Requirements

Implement a manual implementation of backpropagation for a **fully-connected** network. The implementation must support:

- any network depth and any layer size
- $ReLU$, $Identity$ or $Sigmoid$ activation each layer
- $RSS$, $ASE$ or $LogLoss$ loss function
  Given an input $x$ and an output $y$, the implementation will:
- show $\hat{y}$ and the Loss
- show the gradient for $W^{(l)}, b^{(l)}$ for every layer $l$


## Imports


In [138]:
from typing import Protocol
import torch
from dataclasses import dataclass

Tensor = torch.Tensor  # avoid typing `torch.Tensor` at every function decleration


### Helper functions for displaying 1-d and 2-d Tensors with LaTex

For easy readbility, we will implement a `texify(tensor)` function which will format the tensor into a $LaTex$ format


In [ ]:
from IPython.core.getipython import get_ipython

CAN_TEXIFY = True
try:
    from IPython.display import display, Math, Latex


except ImportError:
    CAN_TEXIFY = False


def fmt(x) -> str:
    return f"{x:.4f}".rstrip("0").rstrip(".")


def texify(t: Tensor) -> str:
    if not CAN_TEXIFY:
        return t.__repr__()
    if t.dim() == 1:
        rows = [fmt(x) for x in t]  # column vector
    else:
        rows = [" & ".join(fmt(x) for x in row) for row in t]

    body = (r" \\" + "\n").join(rows)
    return r"\begin{bmatrix}" + "\n" + body + "\n" + r"\end{bmatrix}"


def partial(symb: str, l: int) -> str:
    return rf"\frac{{\partial \mathcal{{L}}}}{{\partial {symb}^{{({l})}}}}"


## Activation functions

We will use a protocol for the activation. Each activation object could activate using `act.f(z)` and get the derivative using `act.df(z)`. We also keep a


In [140]:
class Activation(Protocol):  # Later the `MLP` class could use a `list(Activation)` type
    def f(self, z: Tensor) -> Tensor: ...
    def df(self, z: Tensor) -> Tensor: ...
    def __repr__(self) -> str:
        return type(self).__name__


class ReLU(Activation):
    def f(self, z: Tensor) -> Tensor:
        return z.clamp(min=0)

    def df(self, z: Tensor) -> Tensor:
        return (z > 0).to(z.dtype)


class Identity(Activation):
    def f(self, z: Tensor) -> Tensor:
        return z

    def df(self, z: Tensor) -> Tensor:
        return torch.ones_like(z)


class Sigmoid(Activation):
    def f(self, z: Tensor) -> Tensor:
        return 1 / (1 + torch.exp(-z))

    def df(self, z: Tensor) -> Tensor:
        s = self.f(z)
        return s * (1 - s)


## Loss function

Similarly, we will use a protocol for a Loss function object, with the same two methods as before. Unlike the previous, the Loss class will require both $y$ and $\hat{y}$.


In [141]:
class Loss(Protocol):
    def verify_dim_match(self, y_hat: Tensor, y: Tensor) -> None:
        if y_hat.shape != y.shape:
            raise ValueError(
                f"Expected matching shapes, got {y_hat.shape} and {y.shape}. Go check your dimensions ;)"
            )

    def f(self, y_hat: Tensor, y: Tensor) -> Tensor: ...
    def df(self, y_hat: Tensor, y: Tensor) -> Tensor: ...


class RSS(Loss):
    def f(self, y_hat: Tensor, y: Tensor) -> Tensor:
        self.verify_dim_match(y_hat, y)
        return ((y_hat - y) ** 2).sum()

    def df(self, y_hat: Tensor, y: Tensor) -> Tensor:
        self.verify_dim_match(y_hat, y)
        return 2 * (y_hat - y)


class LogLoss(Loss):  # assumes y is a binary and y_hat in range(0,1)
    def f(self, y_hat: Tensor, y: Tensor) -> Tensor:
        self.verify_dim_match(y_hat, y)
        return -1 * (y * torch.log(y_hat) + (1 - y) * torch.log(1 - y_hat))

    def df(self, y_hat: Tensor, y: Tensor) -> Tensor:
        self.verify_dim_match(y_hat, y)
        return -1 * (y / y_hat) + (1 - y) / (1 - y_hat)


class AbsoluteError(Loss):
    def f(self, y_hat: Tensor, y: Tensor) -> Tensor:
        self.verify_dim_match(y_hat, y)
        return torch.abs(y_hat - y)

    def df(self, y_hat: Tensor, y: Tensor) -> Tensor:
        self.verify_dim_match(y_hat, y)
        return (y_hat - y).sign()

## Designing the Network

### Assumptions

- The network is fully connected
- a 1-d array x input means 1 column vector, a 2-d $X$ input means a Batch where each row is an input vector.


### Helper objects

we will use `@dataclass` object for a friendly way to ensure the inputs and and stores for the network are correctly typed. The network must:

- Know how many layers it has, and which activation is used per which layer (`LayerSpec`)
- Store $z$ and $\sigma(z)$ when if forward passes (`ForwardCache`)
- Store the gradients per each layer when backprop (not strictly required - but the requirements ask to show the grads). (`GradCache`).


In [142]:
@dataclass
class LayerSpec:
    size: int
    activation: Activation


@dataclass
class ForwardCache:
    h_prev: Tensor
    z: Tensor


@dataclass
class GradCache:
    db: Tensor
    dW: Tensor

## Network Layer

Uppon initilizatin, the network will create a `Layer` object for each hidden + the output layer. The layers themselves will initilize $W$ and $b$ at the correct dimentions, store the activation function it shell use, and set its forward and backward cache to `None`.

### Forward method

when forwarding, the layer will require the output of the previus layer (or $x$ if it is the first layer). it will store it's input in the cache, as well as the linear output. It will then apply the linear map according to the input shape - $Wx +b$ for a vector input and $XW^\top+b$ for a batch input, and apply the activatin and return the actiated output for the next layer to use and store.

### Backward

The backward method requires the layer's cache to exist (fail if it dosent) as well as the derivative of the error with respect to itself (`dh`). It will compute the derivative with respect to the linear layer, and set the grads for itself using the formula $\delta^{(l)} (h^({l-1}))^\top$ / $\delta^{(l)} W^{(l)}$


In [143]:
STEP_SIZE = 0.03


class Layer:
    def __init__(self, specs: LayerSpec, in_size: int, l_num: int, g: torch.Generator):
        self.activation = specs.activation
        self.W = torch.randn(specs.size, in_size, generator=g)
        self.b = torch.zeros(specs.size)
        self.l_num = l_num

        self.cache: ForwardCache | None = None
        self.grad: GradCache | None = None

    def forward(self, h_prev) -> Tensor:
        z = (
            self.W @ h_prev + self.b
            if h_prev.dim() == 1
            else h_prev @ self.W.T + self.b
        )
        self.cache = ForwardCache(h_prev, z)
        return self.activation.f(z)

    def backward(self, dh, step=False, print=False) -> Tensor:
        if not self.cache:
            raise RuntimeError("Layer has no forward cahce")
        c = self.cache

        z, h_prev = c.z, c.h_prev
        delta = dh * self.activation.df(z)

        if dh.dim() == 1:  # vector
            dW = torch.outer(delta, h_prev)
            db = delta
            prev_dh = self.W.T @ delta
        else:
            dW = delta.T @ h_prev  # batch
            db = delta.sum(0)
            prev_dh = delta @ self.W

        self.grad = GradCache(db, dW)

        if step:
            self.dW = self.W - STEP_SIZE * dW
            self.b = self.b - STEP_SIZE * db
        # if print:
        #     if CAN_TEXIFY:
        #         display(
        #             Latex(
        #                 rf'$\displaystyle {partial("W", self.l_num)} = {texify(dW)},\;\
        #                     {partial("b", self.l_num)} = {texify(db)}$'
        #             )
        #         )
        #     else:
        #         print(
        #             f"dW^({self.l_num}) = {dW}\n\
        #               db^({self.l_num}) = {db}\n\n"
        #         )
        return prev_dh

    def clear(self):
        self.cache = None
        self.grad = None


In [144]:
class Network:
    def __init__(self, in_size: int, loss: Loss, layers_specs: list[LayerSpec]):
        self.in_size = in_size
        self.layers: list[Layer] = self.make_layers(layers_specs)
        self.depth = len(layers_specs)
        self.loss = loss

        if CAN_TEXIFY:
            print("Initiated the network: ")
            self.print_network()
            print("\n\n\n")
        else:
            print(
                f"Initiated a network with {self.depth - 1} hidden layers sized",
                [l.size for l in layers_specs[0:-1]],
                f"with an input size of {in_size} and output size of {layers_spec[-1].size}.\n\n",
            )

    def make_layers(self, layers_specs: list[LayerSpec]) -> list[Layer]:
        g = torch.Generator().manual_seed(42)
        layers = []
        in_size = self.in_size
        for i, l in enumerate(layers_specs):
            layers.append(Layer(l, in_size, i + 1, g))
            in_size = l.size
        return layers

    def print_network(self):
        def net_layer(i: int) -> str:
            if i < 0:
                return "x"
            l = self.layers[i]
            body = rf'{texify(l.W)}{net_layer(i - 1)} + {texify(l.b)}'
            if isinstance(l.activation, Identity):
                return body

            return rf'{l.activation!r}\left({body}\right)'

        display(Latex(rf'$$ \hat{{y}} = {net_layer(self.depth - 1)} $$'))

    def run(self, x: Tensor, y: Tensor, epocs=1):
        dtype = torch.get_default_dtype()
        x = torch.as_tensor(x, dtype=dtype)
        y = torch.as_tensor(y, dtype=dtype)
        if CAN_TEXIFY:
            display(Latex(rf'Forwarding $x = {texify(x)}, y = {texify(y)}$'))
        else:
            print(f"Running for x = {texify(x)}, output y = {texify(y)}...")

        input = x

        for i in range(epocs):
            input = x
            for l in self.layers:
                output = l.forward(input)
                input = output

            y_hat = input

            dh = self.loss.df(y_hat, y)

            for l in reversed(self.layers):
                dh = l.backward(dh, True)

        y_hat = input
        err = y_hat - y
        loss = self.loss.f(y_hat, y)
        # d_loss = self.loss.df(y_hat, y)

        if CAN_TEXIFY:
            display(
                Latex(
                    rf'$\hat{{y}} = {texify(y_hat)},\;\
                        Error = {texify(err)},\;\
                        \mathcal{{L}} = {fmt(loss)}$'
                )
            )
        else:
            print(f"y_hat={y_hat}\nerror = {err}\nloss = {loss}\n\n")

        # dh = d_loss
        for i, l in enumerate(reversed(self.layers)):
            num = self.depth - i
            # next_dh = l.backward(dh, step=False, print=False)

            if not l.grad:
                raise RuntimeError("Failed to save gradients")

            dW, db = l.grad.dW, l.grad.db

            if CAN_TEXIFY:
                display(
                    Latex(
                        rf'$\displaystyle {partial("W", num)} = {texify(dW)},\;\
                            {partial("b", num)} = {texify(db)}$'
                    )
                )
            else:
                print(
                    f"dW^({num}) = {dW}\n\
                      db^({num}) = {db}\n\n"
                )
            # dh = next_dh
            l.clear()


In [145]:
relu = ReLU()
loss = RSS()
layers_spec = [LayerSpec(size, relu) for size in [3, 3]] + [LayerSpec(2, Identity())]
net = Network(2, loss, layers_spec)

x = torch.tensor([1.0, 1.0])
y = torch.tensor([-1.0, 0.5])
net.run(x, y)


Initiated the network: 


<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

In [146]:
x = torch.tensor([[1.0, 1.0], [0.5, 2]])
y = torch.tensor([[-1.0, 0.5], [1, -1]])
net.run(x, y, epocs=1)

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>